# Chapter 25 — SQLAlchemy, pandas read_sql, and query-to-DataFrame patterns

**Book role:** Databases  
**Suggested book space:** about 9 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Use SQLAlchemy engines as a reusable database abstraction.
- Load parameterized SQL results into DataFrames.
- Push filters/aggregations down to the database where appropriate.
- Define connection and transaction ownership clearly.

## 1. Engines, connections, and sessions

A SQLAlchemy `Engine` manages the dialect and connection pool; a `Connection` represents an active DB-API connection; an ORM `Session` manages ORM identity and unit-of-work behavior. For a reporting app that issues SQL and returns DataFrames, Core connections are often enough. An ORM is useful when application logic genuinely benefits from mapped entity objects and relationships, but it is not required for every database query.

Create the engine once per application configuration, not once per button click, then acquire and release connections for units of work. Keep the connection string out of logs. SQLAlchemy supports many backends, but driver installation, URL format, placeholder semantics, and server behavior still depend on the dialect/driver. Test against the actual target database version.

### Worked example 1: Create an engine and read with parameters

`URL.create` avoids manual URL string encoding errors. The code leaves the real query commented because it requires a configured server and table. A long-lived GUI generally keeps the engine alive and disposes it during shutdown.

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

# Install a matching MySQL driver and set MYSQL_* variables before running.
# Build the URL using URL.create in production so special characters in passwords are encoded.
from sqlalchemy.engine import URL
url = URL.create(
    "mysql+mysqlconnector",
    username=os.getenv("MYSQL_USER", "analytics_reader"),
    password=os.getenv("MYSQL_PASSWORD", ""),
    host=os.getenv("MYSQL_HOST", "127.0.0.1"),
    port=int(os.getenv("MYSQL_PORT", "3306")),
    database=os.getenv("MYSQL_DATABASE", "analytics"),
)
engine = create_engine(url, pool_pre_ping=True, pool_recycle=1800)
query = text("SELECT order_id, order_date, region, revenue FROM orders WHERE order_date >= :start_date AND order_date < :end_date")
# Run only when the configured server and schema are available.
# with engine.connect() as connection:
#     result = pd.read_sql_query(query, connection, params={"start_date": start, "end_date": end})
#     print(result.head())
engine.dispose()  # Use this only when this engine is no longer needed by the application.

## 2. read_sql_query and parameter conventions

`pandas.read_sql_query()` converts SQL query results into a DataFrame. It accepts a query and a connection or SQLAlchemy engine, with parameters passed separately. Parameter syntax is mediated by the driver/dialect, so use the documented parameter style for the chosen integration. Never create a query by concatenating values from search boxes or selection controls.

Use database-side `WHERE`, `GROUP BY`, and `ORDER BY` when transferring fewer rows or aggregating near the source is beneficial. For exploratory analysis, loading a manageable dataset and using pandas may be appropriate. For millions of records, pushing a filter into Python after fetching everything wastes memory and time. Decide based on data volume, latency, query plan, and whether computations can be expressed faithfully in SQL.

### Worked example 2: Push aggregation into SQL

The name `order_items_view` is illustrative. For an actual schema, verify whether order-level values are duplicated across order lines and whether the query counts the right grain.

In [ ]:
QUERY_BY_REGION = """
SELECT region,
       COUNT(DISTINCT order_id) AS order_count,
       SUM(quantity * unit_price) AS revenue
FROM order_items_view
WHERE order_date >= :start_date AND order_date < :end_date
GROUP BY region
ORDER BY revenue DESC
"""
# Compare this grouped result with loading all detail rows and grouping in pandas.
# Choose the location of computation based on data size, freshness, and semantics.
print(QUERY_BY_REGION)

## 3. Connection lifetime and SQL safety

Use context managers to make connection checkout and return clear. For read-only queries, the driver may still start a transaction depending on configuration; understand the behavior if long-running reads can hold locks or snapshots. For writes, explicitly define the commit/rollback boundary. Keep identifiers and sort options on an allow-list if they need to vary dynamically—value parameters do not generally parameterize a column name.

Test SQL with small controlled fixtures and compare DataFrame dtypes and null behavior across backends. SQL NULL, pandas `NaN`/`NA`, timezone handling, decimal precision, and driver conversions can differ. A stable repository contract should normalize these differences before the data reaches widgets or shared analysis functions.

## Guided lab: Lab: interchangeable CSV and SQL source

Implement two adapters that expose `fetch_sales(criteria) -> DataFrame`: one reads the course CSV and one uses SQLAlchemy. Both must return the same normalized columns, dtypes, and documented row grain. Compare their results for a small sample. Include a timeout, resource cleanup, and error mapping to distinguish configuration errors from query errors.

## Review questions and practice

1. What is the difference between Engine and Connection in SQLAlchemy?
2. Why should aggregate queries often run near a large database?
3. Why are dynamic column names treated differently from parameter values?
4. What dtype/null differences can appear between SQL and pandas?

## Further reading

- SQLAlchemy Core: https://docs.sqlalchemy.org/en/20/core/
- pandas read_sql_query: https://pandas.pydata.org/docs/reference/api/pandas.read_sql_query.html

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.